In [ ]:
import os

os.environ['LLM_API_KEY'] = input('Paste your OpenRouter API key: ').strip()
print("API key set:", bool(os.environ.get('LLM_API_KEY')))

In [ ]:
import requests
import json
import time

LLM_MODEL = "openrouter/free"
LLM_URL = "https://openrouter.ai/api/v1/chat/completions"

def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512, max_retries=5):
    api_key = os.environ.get('LLM_API_KEY')

    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    for attempt in range(max_retries):
        response = requests.post(LLM_URL, headers=headers, json=payload)

        if response.status_code == 200:
            time.sleep(2)
            return response.json()['choices'][0]['message']['content']

        if response.status_code == 429:
            retry_after = int(response.headers.get("Retry-After", 15))
            print(f"Rate limited. Waiting {retry_after}s before retry {attempt+1}/{max_retries}...")
            time.sleep(retry_after + 2)
            continue

        print("Error:", response.status_code, response.text)
        return None

    print("Max retries exceeded.")
    return None

In [ ]:
test_output = call_llm(
    system_prompt="You are a helpful assistant.",
    user_prompt="Reply with only the word: hello",
    temperature=0.0
)
print("Test output:", test_output)

In [ ]:
extraction_schema = {
    "type": "object",
    "properties": {
        "customer_name": {"type": "string"},
        "product_name": {"type": "string"},
        "issue_type": {"type": "string"},
        "sentiment": {"type": "string", "enum": ["positive", "neutral", "negative"]},
        "refund_requested": {"type": "boolean"}
    },
    "required": ["customer_name", "product_name", "issue_type", "sentiment", "refund_requested"]
}

In [ ]:
SYSTEM_PROMPT = """You are a structured data extractor. Given a raw customer support message, \
extract exactly these fields and output ONLY valid JSON, with no extra text, no markdown \
formatting, and no explanation:
{
  "customer_name": string,
  "product_name": string,
  "issue_type": string,
  "sentiment": "positive" | "neutral" | "negative",
  "refund_requested": true | false
}"""

FEW_SHOT_EXAMPLES = """Example 1
Input: "Hi, this is Priya. My Bluetooth headphones (SoundMax Pro) stopped charging after two \
days. Very disappointed, I want my money back."
Output: {"customer_name": "Priya", "product_name": "SoundMax Pro", "issue_type": "charging failure", "sentiment": "negative", "refund_requested": true}

Example 2
Input: "Hello, I'm Arjun. Just wanted to say the AeroFit Running Shoes I bought are fantastic, \
great cushioning and fit true to size."
Output: {"customer_name": "Arjun", "product_name": "AeroFit Running Shoes", "issue_type": "none", "sentiment": "positive", "refund_requested": false}
"""

def build_user_prompt(raw_text):
    return f"{FEW_SHOT_EXAMPLES}\nNow extract from this input:\nInput: \"{raw_text}\"\nOutput:"

In [ ]:
import jsonschema
from jsonschema import validate, ValidationError

def extract_structured_data(raw_text, temperature=0.0):
    """
    Calls the LLM, parses its response as JSON, validates against the schema.
    Returns (parsed_dict_or_None, raw_response_string, status_string)
    """
    user_prompt = build_user_prompt(raw_text)
    raw_response = call_llm(SYSTEM_PROMPT, user_prompt, temperature=temperature)

    if raw_response is None:
        return None, None, "fail (no response)"

    cleaned = raw_response.strip()

    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
        return None, raw_response, "fail (invalid JSON)"

    try:
        validate(instance=parsed, schema=extraction_schema)
    except ValidationError as e:
        print(f"Schema validation error: {e.message}")
        fallback = {k: None for k in extraction_schema["required"]}
        return fallback, raw_response, "fail (schema validation)"

    return parsed, raw_response, "pass"

In [ ]:
import re

def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))

def safe_extract(raw_text, temperature=0.0):
    if has_pii(raw_text):
        print("Input blocked: PII detected.")
        return None, None, "blocked"
    return extract_structured_data(raw_text, temperature=temperature)

# --- Guardrail demonstration ---
test_with_pii = "Hi, I'm Rahul, reach me at rahul.k@email.com about my broken toaster."
test_without_pii = "Hi, I'm Rahul, my toaster (HeatWave 3000) stopped working after a week."

print("Test 1 (contains PII):")
result1 = safe_extract(test_with_pii)
print(result1)

print("\nTest 2 (no PII):")
result2 = safe_extract(test_without_pii)
print(result2)

In [ ]:
demo_inputs = [
    "Hello, I'm Neha. The GlowSkin Face Serum I ordered arrived leaking and half empty. I'd like a refund please.",
    "This is Kabir. The ChronoWatch X1 I bought works perfectly, love the battery life!",
    "Hi, Simran here. My order of the UrbanTrek Backpack was delayed by 10 days and the zipper is already broken. Not happy, please refund."
]

demo_results = []

for text in demo_inputs:
    parsed, raw, status = safe_extract(text, temperature=0.0)
    demo_results.append({
        "input": text,
        "llm_output": raw,
        "valid_json": status
    })
    print(f"\nInput: {text}")
    print(f"LLM Output: {raw}")
    print(f"Validation Status: {status}")

In [ ]:
temp_comparison = []

for text in demo_inputs:
    parsed_0, raw_0, status_0 = extract_structured_data(text, temperature=0.0)
    parsed_7, raw_7, status_7 = extract_structured_data(text, temperature=0.7)

    temp_comparison.append({
        "input": text,
        "output_temp_0": raw_0,
        "output_temp_0.7": raw_7
    })

    print(f"\nInput: {text}")
    print(f"Temp=0   -> {raw_0}")
    print(f"Temp=0.7 -> {raw_7}")